# CranioVision — Validation & Confidence Calibration Study

Two questions a clinician needs answered before trusting this system, in one notebook:

**Part A — How well does it perform?**
Per-model and ensemble Dice / HD95 on held-out cases with ground truth, benchmarked
against published BraTS results so the numbers have an external yardstick.

**Part B — Can the system tell when it is likely to be wrong?**
Does cross-model agreement predict segmentation error? If yes, high agreement on a new
patient is evidence the output is reliable — and we can quote a calibrated expectation
rather than an uninterpretable percentage.

> **Prior work backing Part B:** this is an established idea, not a novel one — see
> `research/evidence_agreement_calibration.md` for verified citations (QU-BraTS was a
> whole MICCAI challenge task on exactly this, on exactly this dataset). We are doing
> *product validation*: measuring the effect **on our own checkpoints**.

**Statistical reality check.** With n=10 cases, only a very strong effect is
detectable (|r| ≥ 0.632 for p<0.05; 63% power against a true r=0.7). n=10 is a
**pilot**; a null result would be uninformative. n≈30 makes it adequately powered.
The notebook prints this explicitly with your actual n so the result is never
over-read.

---
## 0. Environment

Runs locally, on Colab, or on Kaggle. Set `REPO_DIR` / `DATA_DIR` / `MODELS_DIR` below
if auto-detection doesn't find them.

In [ ]:
# --- Colab / Kaggle bootstrap (no-op locally) -------------------------------
import os, sys, subprocess
from pathlib import Path

ON_COLAB  = "google.colab" in sys.modules or Path("/content").exists()
ON_KAGGLE = Path("/kaggle").exists()
print(f"Colab={ON_COLAB}  Kaggle={ON_KAGGLE}")

if ON_COLAB or ON_KAGGLE:
    # Only what this notebook needs — no ANTs/reportlab (atlas + PDF not used here).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "monai>=1.5.0", "nibabel", "torch", "scipy", "pandas",
                    "matplotlib", "seaborn"], check=False)

### Google Drive (Colab only)

Mounting keeps a multi-GB dataset inside Google's network — nothing is downloaded to
your machine. Two things that commonly trip this up:

1. **A folder shared *with* you is not directly reachable under the mount.** In Drive,
   open *Shared with me*, right-click the folder → **Organise → Add shortcut to Drive**.
   Only then does it appear under `/content/drive/MyDrive/...`.
2. **Reading thousands of NIfTI files straight off the Drive FUSE mount is slow**
   (high per-file latency). Copying to Colab's local disk first is usually far faster
   overall — see `COPY_TO_LOCAL` below.

Kaggle has no equivalent: `drive.mount` is a `google.colab` API. On Kaggle, attach the
data as a (private) Kaggle Dataset instead and point `CRANIOVISION_DATA` at
`/kaggle/input/...`.

In [ ]:
# --- Mount Drive and point the paths at it (Colab only) ---------------------
MOUNT_DRIVE    = True     # set False if data is already on local disk
COPY_TO_LOCAL  = True     # copy Drive -> /content for much faster file IO

# EDIT THESE to match your Drive layout (after adding a shortcut to My Drive):
DRIVE_DATA   = "/content/drive/MyDrive/BraTS2024_small_dataset"
DRIVE_MODELS = "/content/drive/MyDrive/CranioVision_models"

if ON_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")          # opens an OAuth prompt

    import shutil, time
    src_data = Path(DRIVE_DATA)
    if not src_data.exists():
        raise FileNotFoundError(
            f"{src_data} not found.\n"
            "If this folder was shared with you, add a shortcut first:\n"
            "  Drive -> Shared with me -> right-click -> Organise -> Add shortcut to Drive"
        )

    if COPY_TO_LOCAL:
        local = Path("/content/data/BraTS2024_small_dataset")
        if not local.exists():
            t0 = time.time()
            print(f"Copying {src_data} -> {local} (one-off, speeds up every later read)...")
            shutil.copytree(src_data, local)
            print(f"  done in {(time.time()-t0)/60:.1f} min")
        os.environ["CRANIOVISION_DATA"] = str(local)
    else:
        os.environ["CRANIOVISION_DATA"] = str(src_data)

    if Path(DRIVE_MODELS).exists():
        os.environ["CRANIOVISION_MODELS"] = DRIVE_MODELS

    print("CRANIOVISION_DATA  =", os.environ.get("CRANIOVISION_DATA"))
    print("CRANIOVISION_MODELS=", os.environ.get("CRANIOVISION_MODELS"))
else:
    print("Not mounting Drive (not on Colab, or MOUNT_DRIVE=False).")

In [ ]:
# --- GPU check --------------------------------------------------------------
# Inference is ~5-8x faster on a T4 than on CPU. If this prints False, enable it:
#   Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU
import torch
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory     : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
else:
    print("Running on CPU — expect roughly 3-5 min per case for all three models.")

In [ ]:
# --- Locate repo / data / checkpoints ---------------------------------------
from pathlib import Path
import sys

def _first_existing(cands):
    for c in cands:
        if c and Path(c).exists():
            return Path(c)
    return None

# On Colab/Kaggle the notebook is usually opened on its own, without the repo
# alongside it — so if src/cranovision isn't found, clone it rather than fail.
REPO_URL    = "https://github.com/DPHeshanRanasinghe/CranioVision.git"
REPO_BRANCH = "docs/research"

def _has_src(p):
    return bool(p) and (Path(p) / "src" / "cranovision").exists()

REPO_DIR = _first_existing([
    os.environ.get("CRANIOVISION_REPO"),
    Path.cwd(), Path.cwd().parent,                       # local: research/ or repo root
    "/content/CranioVision", "/kaggle/working/CranioVision",
])
# If we landed in research/, step up to the repo root (where src/ lives).
if REPO_DIR and not _has_src(REPO_DIR) and _has_src(REPO_DIR.parent):
    REPO_DIR = REPO_DIR.parent

if not _has_src(REPO_DIR):
    target = Path("/content/CranioVision") if ON_COLAB else (
        Path("/kaggle/working/CranioVision") if ON_KAGGLE else Path.cwd() / "CranioVision"
    )
    if not _has_src(target):
        print(f"src/cranovision not found — cloning {REPO_BRANCH} into {target} ...")
        r = subprocess.run(
            ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(target)],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            raise RuntimeError(
                f"git clone failed:\n{r.stderr}\n\n"
                "If the repository is private, clone with a token instead:\n"
                "  https://<GITHUB_TOKEN>@github.com/<owner>/<repo>.git\n"
                "or set CRANIOVISION_REPO to a directory that already contains src/."
            )
    REPO_DIR = target

assert _has_src(REPO_DIR), (
    f"Could not find repo root (needs src/cranovision). Got: {REPO_DIR}\n"
    "Set the CRANIOVISION_REPO environment variable to the repo root."
)
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
print("REPO_DIR  :", REPO_DIR)

from src.cranovision.config import (
    RAW_DATA_DIR, MODELS_DIR, OUTPUTS_DIR, DEVICE, MODALITIES, CLASS_NAMES,
)
# Allow overrides for Colab/Kaggle mounted datasets:
DATA_DIR   = Path(os.environ.get("CRANIOVISION_DATA",   RAW_DATA_DIR))
CKPT_DIR   = Path(os.environ.get("CRANIOVISION_MODELS", MODELS_DIR))
RESULTS_DIR = REPO_DIR / "research" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR  :", DATA_DIR, "(exists:", DATA_DIR.exists(), ")")
print("CKPT_DIR  :", CKPT_DIR, "(exists:", CKPT_DIR.exists(), ")")
print("DEVICE    :", DEVICE)
print("RESULTS   :", RESULTS_DIR)

---
## 1. Cases

Only cases **with a ground-truth mask** can be used — both parts need GT to measure error.

In [ ]:
import re

def scan_cases(data_dir: Path):
    """Find BraTS case folders that have all 4 modalities AND a GT seg."""
    cases, skipped = [], []
    for d in sorted(Path(data_dir).iterdir()):
        if not d.is_dir():
            continue
        files = {f.name.lower(): f for f in d.iterdir() if f.suffix in (".nii", ".gz")}
        def find(suffix):
            for n, f in files.items():
                if f"-{suffix}." in n or n.endswith(f"-{suffix}.nii") or f"_{suffix}." in n:
                    return f
            return None
        mods = [find(m) for m in MODALITIES]
        seg  = find("seg")
        if all(mods) and seg:
            cases.append({"case_id": d.name,
                          "image": [str(p) for p in mods],
                          "label": str(seg)})
        else:
            skipped.append((d.name, "missing modality" if not all(mods) else "no GT mask"))
    return cases, skipped

CASES, SKIPPED = scan_cases(DATA_DIR)
print(f"Usable cases (4 modalities + GT): {len(CASES)}")
for cid, why in SKIPPED:
    print(f"  skipped {cid}: {why}")
assert CASES, "No usable cases found — check DATA_DIR."
[c["case_id"] for c in CASES]

### Restrict to the held-out test split

Performance must be measured on cases the models never trained on. The original
developer's evaluation stored its test-set case IDs in `outputs/*_test_results.json`;
if that file is present we reuse exactly that split, so our numbers are comparable to
theirs and free of train/val contamination.

Set `RESTRICT_TO_TEST_SPLIT = False` only for exploratory runs — never for a number
you intend to show a clinician.

In [ ]:
RESTRICT_TO_TEST_SPLIT = True

def load_reference_results():
    """Prior evaluation artifacts from the original developer, if present."""
    ref = {}
    for name in ["attention_unet", "swin_unetr", "nnunet", "ensemble"]:
        p = OUTPUTS_DIR / f"{name}_test_results.json"
        if p.exists():
            ref[name] = json.loads(p.read_text())
    return ref

import json
REFERENCE = load_reference_results()
TEST_IDS = set()
for r in REFERENCE.values():
    for c in r.get("per_case", []):
        TEST_IDS.add(c["case_id"])

print(f"Reference result files found : {list(REFERENCE) or 'none'}")
print(f"Test-split case IDs recovered: {len(TEST_IDS)}")

if TEST_IDS and RESTRICT_TO_TEST_SPLIT:
    kept = [c for c in CASES if c["case_id"] in TEST_IDS]
    dropped = [c["case_id"] for c in CASES if c["case_id"] not in TEST_IDS]
    if kept:
        print(f"\nRestricting to held-out test split: {len(kept)}/{len(CASES)} cases")
        if dropped:
            print(f"  excluded (not in test split): {dropped}")
        CASES = kept
    else:
        print("\n!! None of the local cases are in the stored test split.")
        print("   Keeping all cases, but Part A is NOT a clean held-out estimate.")
elif not TEST_IDS:
    print("\nNo stored test split found — cannot guarantee these are held-out cases.")
    print("Part A should be read as indicative only.")

print(f"\nFinal case count: {len(CASES)}")

In [ ]:
# --- Power check: state up front what this n can and cannot show -------------
import numpy as np
from scipy import stats

def power_report(n, true_r=0.7, alpha=0.05):
    """Fisher-z power. Undefined for n<=3 (the z transform needs n-3>0)."""
    if n <= 4:
        return float("nan"), float("nan")
    df = n - 2
    tcrit = stats.t.ppf(1 - alpha / 2, df)
    rcrit = tcrit / np.sqrt(df + tcrit ** 2)
    z, se, zc = np.arctanh(true_r), 1 / np.sqrt(n - 3), np.arctanh(rcrit)
    power = 1 - stats.norm.cdf((zc - z) / se)
    return rcrit, power

N = len(CASES)
rcrit, power = power_report(N)
print(f"n = {N}")
if np.isnan(rcrit):
    print("  Too few cases for any meaningful correlation analysis (need n>4).")
    print("  >> Part B results below are NOT interpretable at this n.")
else:
    print(f"  minimum |r| for p<0.05 : {rcrit:.3f}")
    print(f"  power vs a true r=0.7  : {power*100:.0f}%")
    if N < 20:
        print("\n  >> PILOT ONLY. A null result here is NOT evidence of no effect.")
        print("  >> Target n>=30 for an adequately powered claim.")
    else:
        print("\n  >> Adequately powered for a strong effect.")

---
## 2. Run the models

One model at a time — loaded, inferred, freed — to respect the 4 GB GPU budget the rest
of the project is designed around (`ensemble_predict` would hold all three at once).
Softmax probabilities move to CPU before ensembling.

Results are cached to `research/results/per_case_raw.json`, so re-running the analysis
cells is instant and you only pay inference once.

In [ ]:
import gc, json, time
import torch

from src.cranovision.pipeline import PIPELINE_MODELS
from src.cranovision.inference.predict import load_model, make_inferer, predict_case
from src.cranovision.training.metrics import compute_case_dice, compute_brats_region_dice

CACHE = RESULTS_DIR / "per_case_raw.json"

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

available = {n: c for n, c in PIPELINE_MODELS.items() if (CKPT_DIR / c["ckpt"]).exists()}
print("Checkpoints found:", list(available) or "NONE")
assert available, f"No checkpoints in {CKPT_DIR}"

In [ ]:
def analyse_case(case, inferer):
    """Run every model on one case; return per-model preds + GT (all CPU tensors)."""
    preds, probs = {}, {}
    gt = None
    for name, cfg in available.items():
        model = load_model(name, CKPT_DIR / cfg["ckpt"], **cfg["build_kwargs"])
        out = predict_case(model, dict(case), inferer=inferer, return_probabilities=True)
        preds[name] = out["pred"]
        probs[name] = out["probs"]            # already CPU
        if gt is None and out["label"] is not None:
            gt = out["label"].squeeze(0).long()
        del model, out
        free()
    return preds, probs, gt


def ensemble_from_probs(probs, weights):
    stacked = None
    for name, p in probs.items():
        w = weights.get(name, 1.0 / len(probs))
        stacked = w * p if stacked is None else stacked + w * p
    return stacked.argmax(dim=0)


def agreement_stats(preds):
    """Same definitions as pipeline.py so the notebook measures what the app shows."""
    stacked = torch.stack(list(preds.values()))
    mode = torch.mode(stacked, dim=0).values
    all_agree = (stacked == mode.unsqueeze(0)).all(dim=0)
    any_tumor = (stacked > 0).any(dim=0)
    n_any = int(any_tumor.sum())
    return {
        "unanimous_fraction": float(all_agree.float().mean()),
        "tumor_region_agreement": float((all_agree & any_tumor).sum()) / n_any if n_any else 0.0,
        "n_models_compared": len(preds),
    }

In [ ]:
FORCE_RERUN = False   # set True to recompute every case from scratch

# The cache is keyed by case_id, and we only ever analyse the cases in CASES.
# Without this, a cache written by an earlier *unrestricted* run would silently
# supply extra cases and the statistics would describe a different cohort than
# the one selected above.
cached = {}
if CACHE.exists() and not FORCE_RERUN:
    cached = {r["case_id"]: r for r in json.loads(CACHE.read_text())}
    print(f"Cache holds {len(cached)} case(s).")

wanted = [c["case_id"] for c in CASES]
todo = [c for c in CASES if c["case_id"] not in cached]
reused = [cid for cid in wanted if cid in cached]
if reused:
    print(f"  reusing {len(reused)} cached: {reused}")
if todo:
    print(f"  computing {len(todo)} new: {[c['case_id'] for c in todo]}")

records = [cached[cid] for cid in wanted if cid in cached]

if todo:
    inferer = make_inferer()
    t0 = time.time()
    # Equal weights: with no outputs/*_history.json present the app itself falls
    # back to equal weighting, so this matches shipped behaviour.
    weights = {n: 1.0 / len(available) for n in available}

    for i, case in enumerate(todo, 1):
        t = time.time()
        preds, probs, gt = analyse_case(case, inferer)
        if gt is None:
            print(f"[{i}/{len(CASES)}] {case['case_id']}: no GT after preprocessing — skipped")
            continue

        ens = ensemble_from_probs(probs, weights)
        rec = {"case_id": case["case_id"], **agreement_stats(preds)}

        for name, pred in {**preds, "ensemble": ens}.items():
            dice = compute_case_dice(pred, gt)
            regions = compute_brats_region_dice(pred, gt)
            rec[f"{name}__mean_dice"] = float(np.mean(dice))
            for cls, d in zip(CLASS_NAMES[1:], dice):
                rec[f"{name}__dice_{cls}"] = float(d)
            for r, d in regions.items():
                rec[f"{name}__dice_{r}"] = float(d)

        rec["gt_tumor_voxels"] = int((gt > 0).sum())
        records.append(rec)
        cached[rec["case_id"]] = rec
        del preds, probs, gt, ens
        free()
        print(f"[{i}/{len(todo)}] {rec['case_id']}: "
              f"ens Dice={rec['ensemble__mean_dice']:.3f} "
              f"agree={rec['tumor_region_agreement']:.3f} "
              f"({time.time()-t:.0f}s)")

    # Persist the union so later runs on other splits still benefit.
    CACHE.write_text(json.dumps(list(cached.values()), indent=2))
    print(f"\nDone in {(time.time()-t0)/60:.1f} min -> {CACHE}")

import pandas as pd
df = pd.DataFrame(records)
assert set(df["case_id"]) <= set(wanted), "df contains cases outside the selected split"
print(f"Analysing {len(df)} cases: {sorted(df['case_id'])}")
df.head()

---
## Part A — How well does it perform?

Per-model and ensemble accuracy with 95% confidence intervals, against published BraTS
baselines. The CI width is as important as the mean: with a small pilot it will be wide,
and saying so is what makes the number trustworthy.

In [ ]:
def mean_ci(vals, n_boot=10000, seed=0):
    v = np.asarray([x for x in vals if np.isfinite(x)], dtype=float)
    if len(v) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
    return v.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

model_order = [*available.keys(), "ensemble"]
rows = []
for name in model_order:
    row = {"model": PIPELINE_MODELS.get(name, {}).get("display", name)}
    for region in ["WT", "TC", "ET"]:
        col = f"{name}__dice_{region}"
        if col in df:
            m, lo, hi = mean_ci(df[col])
            row[region] = f"{m:.3f} [{lo:.3f}–{hi:.3f}]"
    m, lo, hi = mean_ci(df[f"{name}__mean_dice"])
    row["Mean Dice"] = f"{m:.3f} [{lo:.3f}–{hi:.3f}]"
    rows.append(row)

perf = pd.DataFrame(rows)
print(f"Dice, mean [95% bootstrap CI], n={len(df)}\n")
perf

In [ ]:
# --- External yardstick ------------------------------------------------------
# nnU-Net's BraTS 2020 winning entry (Isensee et al.), as reported in
# "nnU-Net for Brain Tumor Segmentation" (arXiv:2011.00848).
# NOTE: different test set + different year -> indicative context, NOT a
# like-for-like comparison. Do not present as a head-to-head result.
BASELINE = {"WT": 0.889, "TC": 0.851, "ET": 0.820, "source": "nnU-Net, BraTS 2020 winner"}

print(f"Reference: {BASELINE['source']}")
print(f"  WT={BASELINE['WT']}  TC={BASELINE['TC']}  ET={BASELINE['ET']}\n")
for region in ["WT", "TC", "ET"]:
    col = f"ensemble__dice_{region}"
    if col in df:
        m, lo, hi = mean_ci(df[col])
        delta = m - BASELINE[region]
        print(f"  {region}: ours {m:.3f} [{lo:.3f}–{hi:.3f}]   "
              f"vs {BASELINE[region]:.3f}  ({delta:+.3f})")
print("\nCaveat: different cohort/protocol. Treat as sanity context, not a ranking.")

### Reproducibility check against the prior evaluation

If we are running the same held-out split as the original developer's stored results,
our per-model numbers should land close to theirs. Large gaps mean something changed —
different preprocessing, different checkpoints, or a different split — and must be
explained before either set of numbers is trusted.

In [ ]:
if REFERENCE:
    rows = []
    for name in model_order:
        ref = REFERENCE.get(name)
        if not ref:
            continue
        ref_overall = ref.get("overall_mean_dice")
        ours = float(np.nanmean(df[f"{name}__mean_dice"]))
        rb = ref.get("brats_regions", {})
        row = {
            "model": name,
            "ours (mean Dice)": f"{ours:.3f}",
            "reference": f"{ref_overall:.3f}" if ref_overall is not None else "—",
            "delta": f"{ours - ref_overall:+.3f}" if ref_overall is not None else "—",
            "n ours": len(df),
            "n ref": ref.get("num_test_cases") or ref.get("n_test_cases"),
        }
        for r in ["WT", "TC", "ET"]:
            if f"{r}_mean" in rb and f"{name}__dice_{r}" in df:
                row[f"{r} ours/ref"] = (
                    f"{np.nanmean(df[f'{name}__dice_{r}']):.3f} / {rb[f'{r}_mean']:.3f}"
                )
        rows.append(row)
    print("Ours vs stored prior evaluation\n")
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nDeltas within ~0.02 on the same split = consistent.")
    print("Larger gaps => investigate before quoting either number.")
else:
    print("No reference results available to compare against.")

### Does the ensemble actually beat the best single model?

Worth checking explicitly, because the prior evaluation found it did **not**
(`ensemble_gain_over_best_single = -0.008` — the ensemble scored marginally *below*
SwinUNETR alone). That is consistent with the project's own framing: the ensemble earns
its place as a **cross-model agreement/confidence signal**, not as a Dice improvement.
If that holds here, say so plainly rather than implying the ensemble is more accurate.

In [ ]:
singles = {n: float(np.nanmean(df[f"{n}__mean_dice"])) for n in available}
ens_score = float(np.nanmean(df["ensemble__mean_dice"]))
best_name = max(singles, key=singles.get)
gain = ens_score - singles[best_name]

for n, v in sorted(singles.items(), key=lambda kv: -kv[1]):
    print(f"  {n:<16} {v:.4f}")
print(f"  {'ensemble':<16} {ens_score:.4f}")
print()
print(f"best single = {best_name} ({singles[best_name]:.4f})")
print(f"ensemble gain over best single = {gain:+.4f}")
print()
if gain <= 0:
    print("=> Ensemble does NOT improve Dice over the best single model.")
    print("   Its value is the agreement/confidence signal (Part B), not accuracy.")
else:
    print("=> Ensemble improves on the best single model here.")

---
## Part B — Does agreement predict accuracy?

**H0:** tumor-region agreement carries no monotonic information about ensemble Dice.

We use **Spearman ρ** (rank-based; no linearity assumption, robust to the small-n
outliers a 10-case pilot will have). Bootstrap CI on ρ, and a permutation test as a
distribution-free check.

In [ ]:
X_COL, Y_COL = "tumor_region_agreement", "ensemble__mean_dice"
x, y = df[X_COL].values, df[Y_COL].values

rho, p_rho = stats.spearmanr(x, y)
r_p, p_p   = stats.pearsonr(x, y)

# bootstrap CI on Spearman rho
rng = np.random.default_rng(0)
idx = rng.integers(0, len(x), size=(10000, len(x)))
boot = np.array([stats.spearmanr(x[i], y[i]).statistic for i in idx])
boot = boot[np.isfinite(boot)]
lo, hi = np.percentile(boot, [2.5, 97.5])

# permutation test (distribution-free)
perm = np.array([stats.spearmanr(rng.permutation(x), y).statistic for _ in range(10000)])
p_perm = float((np.abs(perm) >= abs(rho)).mean())

print(f"n = {len(x)}")
print(f"Spearman rho = {rho:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  p = {p_rho:.4f}")
print(f"Pearson  r   = {r_p:+.3f}  p = {p_p:.4f}")
print(f"Permutation p = {p_perm:.4f}")
print()
if p_rho < 0.05 and rho > 0:
    print("=> Higher agreement is associated with higher Dice (significant at this n).")
elif rho > 0:
    print("=> Positive trend, NOT significant at this n.")
    print("   With this sample size that is expected even if the effect is real —")
    print("   see the power report above. Collect more cases before concluding.")
else:
    print("=> No positive association found. Do NOT ship the confidence framing on this evidence.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sc = ax.scatter(x, y, s=90, c=df["gt_tumor_voxels"], cmap="viridis",
                edgecolor="k", linewidth=0.6, zorder=3)
if len(x) > 2:
    b, a = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, a + b * xs, "--", color="crimson", lw=2,
            label=f"fit (Spearman ρ={rho:+.2f}, p={p_rho:.3f})")
    ax.legend(loc="lower right", fontsize=9)
ax.set_xlabel("Tumor-region agreement")
ax.set_ylabel("Ensemble mean Dice")
ax.set_title(f"Agreement vs accuracy (n={len(x)})")
ax.grid(alpha=0.3)
plt.colorbar(sc, ax=ax, label="GT tumor voxels")

ax = axes[1]
ax.hist(boot, bins=50, color="steelblue", alpha=0.85)
ax.axvline(rho, color="crimson", lw=2, label=f"observed ρ={rho:+.2f}")
ax.axvline(0, color="k", lw=1, ls=":")
ax.axvspan(lo, hi, color="crimson", alpha=0.12, label="95% CI")
ax.set_xlabel("Bootstrapped Spearman ρ")
ax.set_ylabel("frequency")
ax.set_title("Uncertainty on the correlation")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "agreement_vs_dice.png", dpi=150, bbox_inches="tight")
plt.show()

### The confound: is agreement just a proxy for tumor size?

Large, well-circumscribed tumors are both **easier to segment** and **easier to agree
on**. If the correlation disappears once tumor volume is controlled for, then agreement
is reporting case difficulty rather than model confidence — still operationally useful
(it predicts), but it must be described honestly.

In [ ]:
def partial_spearman(x, y, z):
    """Spearman correlation of x,y controlling for z (rank residual method)."""
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    def resid(a, b):
        b1, b0 = np.polyfit(b, a, 1)
        return a - (b0 + b1 * b)
    ex, ey = resid(rx, rz), resid(ry, rz)
    return stats.pearsonr(ex, ey)

z = df["gt_tumor_voxels"].values.astype(float)
pr, pp = partial_spearman(x, y, z)
rho_xz, p_xz = stats.spearmanr(x, z)
rho_yz, p_yz = stats.spearmanr(y, z)

attenuation = (1 - abs(pr) / abs(rho)) * 100 if rho else float("nan")

print(f"raw Spearman(agreement, dice)            = {rho:+.3f} (p={p_rho:.4f})")
print(f"partial, controlling for tumor volume    = {pr:+.3f} (p={pp:.4f})")
print(f"attenuation                              = {attenuation:.0f}%")
print()
print(f"  agreement vs tumor volume  rho={rho_xz:+.3f} (p={p_xz:.4f})   <-- collinearity")
print(f"  dice      vs tumor volume  rho={rho_yz:+.3f} (p={p_yz:.4f})")
print()

# Judge on SIGNIFICANCE first, not magnitude alone: a partial r can look
# respectable and still be indistinguishable from zero at small n.
if pp < 0.05 and attenuation < 50:
    print("=> Survives the control: agreement carries information beyond tumor size.")
elif pp < 0.05:
    print("=> Attenuated but still significant: partly, not wholly, a size effect.")
else:
    print("=> NOT significant after controlling for tumor volume.")
    print("   Cannot claim agreement is informative independently of tumor size.")
    if abs(rho_xz) > 0.8:
        print(f"   Agreement and tumor volume are highly collinear (rho={rho_xz:+.2f}),")
        print("   so at this n the two explanations are statistically inseparable.")
    print("   -> Operationally the raw correlation still predicts Dice, which is what")
    print("      the UI needs. But describe it as a reliability indicator, NOT as")
    print("      evidence of independent 'model confidence', until n is larger.")

### Calibration table — what the UI would actually say

Correlation alone doesn't help a clinician. This maps an agreement value to an
**expected Dice**, which is the sentence we can put on screen.

In [ ]:
edges = [0.0, 0.70, 0.80, 0.85, 0.90, 1.01]
labels = ["<70%", "70–80%", "80–85%", "85–90%", "≥90%"]
df["_bin"] = pd.cut(df[X_COL], bins=edges, labels=labels, right=False)

cal = []
for b in labels:
    sub = df[df["_bin"] == b]
    if len(sub) == 0:
        cal.append({"Agreement": b, "n": 0, "Observed Dice": "—"})
        continue
    m, lo_, hi_ = mean_ci(sub[Y_COL])
    cal.append({"Agreement": b, "n": len(sub),
                "Observed Dice": f"{m:.3f}",
                "95% CI": f"[{lo_:.3f}–{hi_:.3f}]" if len(sub) > 1 else "n=1"})
cal_df = pd.DataFrame(cal)
print("Calibration — expected ensemble Dice by agreement band\n")
print(cal_df.to_string(index=False))
print("\nBands with n<5 are not reportable to clinicians; collect more cases.")
cal_df

---
## 5. Export

Writes two things:

1. `research/results/validation_summary.json` — the machine-readable record.
2. `outputs/{model}_history.json` — **fixes a live bug**: `pipeline.py` falls back to
   weight `1.0` per model when these are missing, silently turning the documented
   *Dice-weighted* ensemble into plain equal averaging. Writing measured Dice here
   restores the intended behaviour.

> Caveat on (2): these Dice values come from whatever cases this notebook ran. If that
> is the pilot set, they are **not** a clean validation estimate — regenerate on a
> proper held-out split before relying on them for weighting.

In [ ]:
summary = {
    "n_cases": int(len(df)),
    "cases": df["case_id"].tolist(),
    "device": str(DEVICE),
    "models": list(available.keys()),
    "part_a_performance": {
        name: {
            "mean_dice": float(np.nanmean(df[f"{name}__mean_dice"])),
            **{r: float(np.nanmean(df[f"{name}__dice_{r}"]))
               for r in ["WT", "TC", "ET"] if f"{name}__dice_{r}" in df}
        } for name in model_order
    },
    "part_b_calibration": {
        "x": X_COL, "y": Y_COL,
        "spearman_rho": float(rho), "spearman_p": float(p_rho),
        "spearman_ci95": [float(lo), float(hi)],
        "permutation_p": float(p_perm),
        "partial_rho_controlling_volume": float(pr),
        "partial_p": float(pp),
        "min_detectable_r_at_this_n": float(rcrit),
        "power_vs_true_r_0.7": float(power),
        "adequately_powered": bool(len(df) >= 20),
    },
    "baseline_reference": BASELINE,
}
(RESULTS_DIR / "validation_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary["part_b_calibration"], indent=2))

In [ ]:
# --- Persist results off the ephemeral Colab disk ---------------------------
# /content is wiped when the session ends or disconnects. Copy the artifacts
# back to Drive so a 20-minute run is not lost to an idle timeout.
if ON_COLAB and MOUNT_DRIVE:
    import shutil
    dest = Path("/content/drive/MyDrive/CranioVision_results")
    dest.mkdir(parents=True, exist_ok=True)
    for f in RESULTS_DIR.iterdir():
        if f.is_file():
            shutil.copy2(f, dest / f.name)
            print("saved ->", dest / f.name)
    print(f"\nResults persisted to {dest}")
else:
    print(f"Results are in {RESULTS_DIR} (local disk).")

In [ ]:
WRITE_HISTORY = False   # flip to True to restore Dice-weighted ensembling

if WRITE_HISTORY:
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    for name in available:
        best = float(np.nanmean(df[f"{name}__mean_dice"]))
        p = OUTPUTS_DIR / f"{name}_history.json"
        p.write_text(json.dumps({
            "best_dice": best,
            "source": "validation_study.ipynb",
            "n_cases": int(len(df)),
            "warning": "pilot estimate — regenerate on a held-out split before clinical use",
        }, indent=2))
        print(f"wrote {p.name}: best_dice={best:.4f}")
else:
    print("WRITE_HISTORY=False — ensemble stays on equal weights.")
    print("Set True once these Dice values come from a proper held-out split.")

---
## 6. Reading the result honestly

| Outcome | What to tell a clinician |
|---|---|
| ρ > 0, significant, survives volume control | "High agreement indicates a more reliable output — calibrated on n cases." |
| ρ > 0, significant, **fails** volume control | "Agreement tracks case difficulty (tumor size)." Still useful, different claim. |
| ρ > 0, not significant at n<20 | **Nothing yet.** Under-powered. Collect to n≈30 and re-run. |
| ρ ≈ 0 or negative at n≥30 | The signal doesn't work. Don't ship the confidence framing. |

**Two limits that hold regardless of the numbers:**

1. Agreement is a *consensus* signal, not correctness — three models can agree and all
   be wrong. Architectural diversity makes that less likely; it does not make it
   impossible.
2. Everything here is measured on BraTS-format, skull-stripped, co-registered,
   pre-operative adult glioma. It does not transfer to pediatric, meningioma,
   post-operative, or raw clinical acquisitions without re-validation.